#Powered by [@CoinNoin](https://www.youtube.com/@CoinNoin)
[![Subscribe](https://img.shields.io/badge/YouTube-Subscribe%20@CoinNoin-red?style=for-the-badge&logo=youtube)](https://www.youtube.com/@CoinNoin)

In [ ]:
#@title 1. Initialize Core Environment
#@markdown This prepares the environment and installs required libraries for MiniCPM5.

import os
import subprocess
from IPython.display import clear_output

print("🚀 [@CoinNoin] Initializing Core Architecture...")
print("📦 [@CoinNoin] Installing Dependencies (This takes a moment)...")
# Install/upgrade required packages. Colab already provides PyTorch.
!pip install -q -U "transformers>=5.6" accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

clear_output()
print("✅ [@CoinNoin] Environment Ready!")

In [ ]:
#@title 2. Download and Load MiniCPM5 Model
#@markdown Choose the model size. It will be downloaded (if not cached) and loaded into the GPU's memory.

MODEL_NAME = "MiniCPM5-1B" #@param ["MiniCPM5-2B", "MiniCPM5-1B"]

MODEL_IDS = {
    "MiniCPM5-2B": "openbmb/MiniCPM5-2B",
    "MiniCPM5-1B": "openbmb/MiniCPM5-1B",
}

model_id = MODEL_IDS[MODEL_NAME]
print(f"⚡ [@CoinNoin] Loading {model_id}...")
print(f"   CUDA available: {torch.cuda.is_available()}")

# Use fp16 on GPU, fp32 on CPU. This avoids bfloat16 issues on older Colab GPUs like T4.
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"📥 [@CoinNoin] Fetching Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True,
)

print(f"📥 [@CoinNoin] Fetching and Loading Model into VRAM (This may take a few minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()

clear_output()
print(f"✅ [@CoinNoin] {MODEL_NAME} successfully loaded and ready for inference!")

In [ ]:
#@title 3. MiniCPM5 Text Generation
#@markdown Enter your prompt and tweak the generation parameters. Because the model is already loaded in Cell 2, you can run this cell as many times as you want instantly.

PROMPT = "Who are you? Please briefly introduce yourself." #@param {type:"string"}
USE_THINKING = False #@param {type:"boolean"}

#@markdown ---
#@markdown ### ⚙️ Advanced Settings
MAX_NEW_TOKENS = 256 #@param {type:"slider", min:64, max:4096, step:64}
REPETITION_PENALTY = 1.0 #@param {type:"slider", min:1.0, max:2.0, step:0.05}
TEMPERATURE = "auto" #@param ["auto", "0.7", "0.9", "1.0"] {allow-input: true}
#@markdown *Leave TEMPERATURE on `auto` to use the model's recommended default, or type a custom float.*

import torch

print(f"\n\033[94m➜ [@CoinNoin] Generating | Model: {MODEL_NAME} | Thinking: {USE_THINKING} | Max Tokens: {MAX_NEW_TOKENS}\033[0m")

# Recommended sampling per model
if TEMPERATURE == "auto":
    if MODEL_NAME == "MiniCPM5-1B":
        temperature = 0.9 if USE_THINKING else 0.7
    else:
        temperature = 1.0
else:
    temperature = float(TEMPERATURE)

top_p = 0.95
min_p = 0.0

messages = [
    {"role": "user", "content": PROMPT}
]

print("🧠 [@CoinNoin] Preparing prompt and thinking tokens...")
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=USE_THINKING,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

generate_kwargs = dict(
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=True,
    temperature=temperature,
    top_p=top_p,
    repetition_penalty=REPETITION_PENALTY,
)

print("⏳ [@CoinNoin] Generating response (this might take a moment depending on the prompt)...\n")
try:
    outputs = model.generate(
        **inputs,
        min_p=min_p,
        **generate_kwargs,
    )
except TypeError as e:
    if "min_p" in str(e):
        outputs = model.generate(
            **inputs,
            **generate_kwargs,
        )
    else:
        raise

new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
text = tokenizer.decode(new_tokens, skip_special_tokens=True)

print("=" * 80)
print(f"✨ [@CoinNoin] RESPONSE:\n")
print(text)
print("=" * 80)